<a href="https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1:
The research paper reports that AI-driven search behavior is associated with changes in content discovery and user interaction.

Methodology question:
The label definition should be checked carefully. It is important to understand whether the outcome is based on observed user behavior, search engine metrics, or another proxy. The validation design should match the exact claim being made.

Finding 2:
The research paper reports patterns between content characteristics and search performance.

Methodology question:
The validation approach should confirm that the relationship is measured on unseen data and does not include future information. A stronger claim requires careful split design and leakage checks.

Overall:
The findings are useful as directional observations, but methodology details such as label creation, sampling, and validation design determine how broadly the conclusions can be applied.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

The Week-5 model used a random split. For this audit, I use a time-aware split based on content_age_days.

This provides a stricter evaluation because the model is tested on a different time-like portion of the data instead of randomly mixed examples.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position"
]


X = df[features].fillna(0)

y = (df["trend_direction"] == "down").astype(int)


# Before: random split (Week-5 style)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


rf_random = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_random.fit(X_train, y_train)

random_accuracy = accuracy_score(
    y_test,
    rf_random.predict(X_test)
)


# After: time-aware split

df_sorted = df.sort_values("content_age_days")

split = int(len(df_sorted)*0.8)

train = df_sorted.iloc[:split]
test = df_sorted.iloc[split:]


X_train_time = train[features].fillna(0)
X_test_time = test[features].fillna(0)

y_train_time = (train["trend_direction"]=="down").astype(int)
y_test_time = (test["trend_direction"]=="down").astype(int)


rf_time = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_time.fit(
    X_train_time,
    y_train_time
)


time_accuracy = accuracy_score(
    y_test_time,
    rf_time.predict(X_test_time)
)


results = pd.DataFrame({
    "Split": [
        "Random split",
        "Time-aware split"
    ],
    "Accuracy": [
        random_accuracy,
        time_accuracy
    ]
})


results


,Split,Accuracy
0,Random split,0.692167
1,Time-aware split,0.590000


## 3. Leakage audit

I checked the final feature set for fields that contain target information, future windows, or manually created outputs.

These fields are excluded because they would allow the model to access information that would not be available at prediction time.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]


print("Leakage audit:")

for col in leakage_candidates:
    if col in features:
        print("WARNING:", col)
    else:
        print("OK excluded:", col)


Leakage audit:
OK excluded: trend_direction
OK excluded: trend_pct
OK excluded: impressions_last_30d
OK excluded: clicks_last_30d
OK excluded: sessions_last_30d


## 4. Claim rewrite

Original claim:
"The model can identify pages that will improve after optimization."

Safer rewrite:
"The model identifies observed historical patterns associated with content performance changes. The output should be used as decision support for prioritizing pages for review, not as a guarantee of future improvement."


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

* [x] Every section above is filled — markdown thinking AND the code that backs it
* [x] The notebook runs top to bottom with no errors (Runtime → Run all)
* [x] No client names, URLs, or private queries anywhere
* [x] My claims use careful words: observed, measured, directional, decision-support
* [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [7]:
import os
import subprocess

REPO_URL = "https://github.com/Ahmed-coder874/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)